# IS477 - Data Cleaning & Integration
**Project:** Do Critics Matter? Analyzing the Relationship Between Critic Scores and Movie Popularity  
**Datasets:** `datasets/movie_info.csv` (Rotten Tomatoes) + `datasets/tmdb_movies.csv` (TMDB API)  
**Author:** Flynn Huynh

## 1. Imports & Setup

In [23]:
import pandas as pd
import re
import os

## 2. Load Raw Data

In [ ]:
rt_df = pd.read_csv('datasets/movie_info.csv')
tmdb_df = pd.read_csv('datasets/tmdb_movies.csv')

print('RT raw shape:', rt_df.shape)
print('TMDB raw shape:', tmdb_df.shape)

RT raw shape:    (12413, 5)
TMDB raw shape:  (700, 7)


In [25]:
# Check missing values and duplicates
print('RT missing values:')
print(rt_df.isnull().sum())
print('\nRT duplicate titles:', rt_df['title'].duplicated().sum())

RT missing values:
title                0
url                  0
release_date        13
critic_score      3036
audience_score    1587
dtype: int64

RT duplicate titles: 1740


In [26]:
# Same for TMDB
print('TMDB missing values:')
print(tmdb_df.isnull().sum())
print('\nTMDB duplicate IDs:', tmdb_df['id'].duplicated().sum())

TMDB missing values:
id              0
title           0
release_date    2
vote_average    0
vote_count      0
popularity      0
overview        0
dtype: int64

TMDB duplicate IDs: 101


## 3. Shared Utility: Title Normalization

In [27]:
# function to normalize title (Lowercase, get rid of punctuation and extra whitespace)
def normalize_title(title):
    if pd.isna(title):
        return None
    title = title.strip().lower()
    title = re.sub(r'[^\w\s]', '', title)  # remove punctuation
    title = re.sub(r'\s+', ' ', title)      # collapse extra spaces
    return title.strip()

## 4. Cleaning Rotten Tomatoes Dataset

In [28]:
# Strip % and cast scores to numeric
rt_df['critic_score'] = pd.to_numeric(
    rt_df['critic_score'].str.replace('%', '', regex=False), errors='coerce'
)
rt_df['audience_score'] = pd.to_numeric(
    rt_df['audience_score'].str.replace('%', '', regex=False), errors='coerce'
)

In [29]:
# Extract release_year from release_date
# Handles both formats: 'Released Dec 16, 1970' and bare '1970'
def extract_year(val):
    if pd.isna(val):
        return None
    match = re.search(r'\b(19|20)\d{2}\b', str(val))
    return int(match.group()) if match else None

rt_df['release_year'] = rt_df['release_date'].apply(extract_year)

In [30]:
# Normalize title for joining
rt_df['title_clean'] = rt_df['title'].apply(normalize_title)

In [31]:
# Drop rows missing critic_score or release_year (required for analysis)
rt_before = len(rt_df)
rt_clean = rt_df.dropna(subset=['critic_score', 'release_year', 'title_clean']).copy()
print(f'Rows dropped (missing values): {rt_before - len(rt_clean)}')

Rows dropped (missing values): 3114


In [32]:
# Deduplicate: keep entry with highest critic_score per title + year
rt_before_dedup = len(rt_clean)
rt_clean = rt_clean.sort_values('critic_score', ascending=False)
rt_clean = rt_clean.drop_duplicates(subset=['title_clean', 'release_year'], keep='first')
print(f'Rows dropped (duplicates): {rt_before_dedup - len(rt_clean)}')
print(f'RT clean shape: {rt_clean.shape}')

Rows dropped (duplicates): 1194
RT clean shape: (8105, 7)


In [33]:
rt_clean[['title', 'title_clean', 'release_year', 'critic_score', 'audience_score']].head()

,title,title_clean,release_year,critic_score,audience_score
1449,Alambrista,alambrista,1977.0,100.0,78.0
2082,Babylon,babylon,2019.0,100.0,46.0
3349,The Lonely Passion of Judith Hearne,the lonely passion of judith hearne,1987.0,100.0,77.0
238,Taking Off,taking off,1971.0,100.0,84.0
2912,Dreamchild,dreamchild,1985.0,100.0,80.0


## 5. Clean TMDB Dataset

In [34]:
# Parse release_year from 'YYYY-MM-DD' format
tmdb_df['release_year'] = pd.to_datetime(
    tmdb_df['release_date'], errors='coerce'
).dt.year

# Normalize title for joining
tmdb_df['title_clean'] = tmdb_df['title'].apply(normalize_title)

In [35]:
# Drop rows missing key fields
tmdb_before = len(tmdb_df)
tmdb_clean = tmdb_df.dropna(subset=['title_clean', 'release_year', 'popularity']).copy()
print(f'Rows dropped (missing values): {tmdb_before - len(tmdb_clean)}')

# Deduplicate on TMDB id
tmdb_before_dedup = len(tmdb_clean)
tmdb_clean = tmdb_clean.drop_duplicates(subset=['id'], keep='first')
print(f'Rows dropped (duplicate TMDB IDs): {tmdb_before_dedup - len(tmdb_clean)}')
print(f'TMDB clean shape: {tmdb_clean.shape}')

Rows dropped (missing values): 2
Rows dropped (duplicate TMDB IDs): 101
TMDB clean shape: (597, 9)


In [36]:
tmdb_clean[['title', 'title_clean', 'release_year', 'popularity', 'vote_count', 'vote_average']].head()

,title,title_clean,release_year,popularity,vote_count,vote_average
0,Love Story,love story,1970.0,5.2652,725,6.828
1,Airport,airport,1970.0,2.7014,431,6.454
2,M*A*S*H,mash,1970.0,3.8311,1122,6.954
3,Patton,patton,1970.0,2.8816,1182,7.500
4,The Aristocats,the aristocats,1970.0,7.1648,5303,7.295


## 6. Data Integration (Merge on `title_clean` + `release_year`)

In [37]:
merged_df = pd.merge(
    rt_clean,
    tmdb_clean[['title_clean', 'release_year', 'id', 'popularity', 'vote_count', 'vote_average', 'overview']],
    on=['title_clean', 'release_year'],
    how='inner'
)

print(f'Merged shape: {merged_df.shape}')
print(f'Match rate vs TMDB clean:  {len(merged_df) / len(tmdb_clean) * 100:.1f}%')

Merged shape: (255, 12)
Match rate vs TMDB clean:  42.7%


In [38]:
print('Missing values in merged dataset:')
print(merged_df.isnull().sum())

Missing values in merged dataset:
title             0
url               0
release_date      0
critic_score      0
audience_score    3
release_year      0
title_clean       0
id                0
popularity        0
vote_count        0
vote_average      0
overview          0
dtype: int64


In [39]:
# Descriptive statistics on key analysis columns
merged_df[['critic_score', 'popularity', 'vote_count', 'vote_average']].describe()

,critic_score,popularity,vote_count,vote_average
count,255.000000,255.000000,255.000000,255.000000
mean,67.772549,1.840216,589.427451,6.403988
std,23.691489,3.415909,1685.088837,0.736093
min,0.000000,0.158900,3.000000,4.106000
25%,55.500000,0.549800,57.500000,5.931500
50%,73.000000,0.991200,147.000000,6.495000
75%,86.000000,2.025850,525.000000,6.911500
max,100.000000,41.089200,22512.000000,8.687000


In [40]:
merged_df[['title', 'release_year', 'critic_score', 'popularity', 'vote_count']].head(10)

,title,release_year,critic_score,popularity,vote_count
0,Taking Off,1971.0,100.0,0.9118,102
1,The Big Doll House,1971.0,100.0,1.0223,79
2,The New Land,1972.0,100.0,0.8255,104
3,The Go-Between,1971.0,100.0,0.9497,153
4,On a Clear Day You Can See Forever,1970.0,100.0,0.6713,62
5,Days and Nights in the Forest,1970.0,100.0,0.2301,36
6,Lovers and Other Strangers,1970.0,100.0,0.4644,23
7,I Never Sang for My Father,1970.0,100.0,0.3947,39
8,Fat City,1972.0,100.0,0.6838,173
9,The Big Bird Cage,1972.0,100.0,0.7770,83


## 7. Save Cleaned and Merged Datasets

In [41]:
os.makedirs('datasets', exist_ok=True)

rt_clean.to_csv('datasets/rt_clean.csv', index=False)
tmdb_clean.to_csv('datasets/tmdb_clean.csv', index=False)
merged_df.to_csv('datasets/merged_movies.csv', index=False)

print(f'Saved datasets/rt_clean.csv({len(rt_clean)} rows)')
print(f'Saved datasets/tmdb_clean.csv({len(tmdb_clean)} rows)')
print(f'Saved datasets/merged_movies.csv({len(merged_df)} rows)')

Saved datasets/rt_clean.csv(8105 rows)
Saved datasets/tmdb_clean.csv(597 rows)
Saved datasets/merged_movies.csv(255 rows)
